# 05 — Condition essentiality predictions (no validation metrics)

## Scientific question
Which genes does iMO1056 predict as growth-essential under each
**implemented** medium (MOPS-pyruvate, MOPS-succinate, `LB_approx`)?

## Explicit non-goals
- **No** confusion matrices / precision / recall / F1 / MCC
- **No** comparison to experimental labels in this notebook
- Six phenotype classes stay distinct when labels are later joined (V-01 gated)

Logic: `src.gene_essentiality.analyze_essentiality_for_medium` +
`src.medium.apply_medium`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import cobra

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.medium import iter_implemented_media
from src.gene_essentiality import (
    analyze_essentiality_for_medium,
    save_essentiality_results,
)

MODEL = ROOT / 'models' / 'paeruginosa_MODEL1507180020.xml'
OUT = ROOT / 'results' / 'essentiality'
OUT.mkdir(parents=True, exist_ok=True)
THRESHOLD = 0.01  # provisional; see DECISIONS D-09
PROCESSES = 1     # set >1 only if stable on this machine


In [ ]:
model = cobra.io.read_sbml_model(str(MODEL))
assert 'PA_Biomass6_DM' in str(model.objective.expression)
media = [s.medium_id for s in iter_implemented_media()]
media


## Per-medium single-gene deletion
Prefer the scripted runner for long jobs:
`scripts/run_condition_essentiality.py`.
Uncomment the loop below to run inside the notebook (~minutes–tens of minutes).


In [ ]:
RUN_HERE = False  # set True to execute deletions in-notebook

summary_rows = []
if RUN_HERE:
    for mid in media:
        print('===', mid, '===')
        df = analyze_essentiality_for_medium(
            model, mid, threshold_ratio=THRESHOLD, processes=PROCESSES
        )
        path = OUT / f'gene_essentiality_{mid}.csv'
        save_essentiality_results(df, str(path), condition_name=mid)
        summary_rows.append({
            'medium_id': mid,
            'threshold_ratio': THRESHOLD,
            'n_genes': len(df),
            'n_essential': int(df['is_essential'].sum()),
            'wt_growth': float(df['wildtype_growth'].iloc[0]),
            'output_file': str(path.relative_to(ROOT)),
        })
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(OUT / 'condition_essentiality_summary.csv', index=False)
    display(summary)
else:
    print('RUN_HERE=False — use scripts/run_condition_essentiality.py')
    print('Expected outputs:')
    for mid in media:
        print(' ', OUT / f'gene_essentiality_{mid}.csv')


## Load saved prediction tables (if present)


In [ ]:
frames = []
for mid in media:
    p = OUT / f'gene_essentiality_{mid}.csv'
    if p.exists():
        df = pd.read_csv(p)
        frames.append(df)
        print(mid, 'essential', int(df['is_essential'].astype(str).str.lower().eq('true').sum())
              if df['is_essential'].dtype == object
              else int(df['is_essential'].sum()),
              '/', len(df))
    else:
        print('MISSING', p)
if frames:
    display(pd.read_csv(OUT / 'condition_essentiality_summary.csv')
            if (OUT / 'condition_essentiality_summary.csv').exists()
            else 'no summary yet')


## Limitations / what we do **not** claim
- Predictions are model knockouts under reconstructed media — **not** validated
  against experiments yet.
- Default threshold = 1% of WT growth (D-09); sensitivity not done here.
- `LB_approx` essentiality is a sensitivity case only.
- Do **not** compute or report validation metrics until Phase 3 gate opens
  (`VALIDATION_LABEL_MEDIA_OVERLAP.md` / V-01).
- Teaching nb03 simulated metrics remain forbidden.
